In [2]:
import pandas as pd
import icartt
import os
import warnings
import re
from datetime import datetime
import csv
from datetime import datetime, timedelta
from netCDF4 import Dataset
import numpy as np
from scipy import stats
import glob
from math import pi
import ast

In [3]:
# Read the CSV file
path_to_track = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\master_comprehensive.csv"
df = pd.read_csv(path_to_track)

# Show all unique campaign values
unique_campaigns = df['Campaign'].unique()
print(f"All unique campaign values: {unique_campaigns}")

# Define the desired campaign order
campaign_order = [
    'FIREXAQ', 'SEAC4RS', 'CAMP2Ex', 'ASIA-AQ', 'DC3', 'DISCOVERAQ-DC', 
    'NAAMES(2016)', 'DISCOVERAQ-California', 'NAAMES(2017)', 'DISCOVERAQ-Texas', 
    'NAAMES(2015)', 'ACE-ENA', 'ISDAC', 'GOAMAZON', 'BBOP', 'ACMEV', 
    'CACTI', 'TCAP2013', 'TCAP2012', 'CARES', 'CALNEX', 'WECAN'
]

# Convert Date column from YYYYMMDD to datetime for processing
df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')

# Group by Campaign and find min/max dates
date_ranges = df.groupby('Campaign')['Date'].agg(['min', 'max']).reset_index()

# Format dates as DD MONTH_NAME YYYY
date_ranges['Start_Date'] = date_ranges['min'].dt.strftime('%d %B %Y')
date_ranges['End_Date'] = date_ranges['max'].dt.strftime('%d %B %Y')

# Create a clean output with Campaign and formatted date ranges
result = date_ranges[['Campaign', 'Start_Date', 'End_Date']].copy()
result['Date_Range'] = result['Start_Date'] + ' - ' + result['End_Date']

# Reorder campaigns according to specified order
# Create a mapping for sorting
campaign_order_map = {campaign: i for i, campaign in enumerate(campaign_order)}
result['order'] = result['Campaign'].map(campaign_order_map)

# Sort by the order (campaigns not in the list will have NaN and appear at the end)
result = result.sort_values('order').drop('order', axis=1).reset_index(drop=True)

# Display results
print("\nDate ranges for each campaign (in specified order):")
print(result[['Campaign', 'Date_Range']])

print("\nDetailed date ranges (in specified order):")
print(result)

All unique campaign values: ['CAMP2Ex' 'DISCOVERAQ-DC' 'NAAMES(2016)' 'NAAMES(2015)'
 'DISCOVERAQ-California' 'SEAC4RS' 'ASIA-AQ' 'DC3' 'NAAMES(2017)'
 'DISCOVERAQ-Texas' 'FIREXAQ' 'CALNEX' 'WECAN' 'BBOP' 'TCAP2013' 'CACTI'
 'ACE-ENA' 'ACMEV' 'TCAP2012' 'ISDAC' 'GOAMAZON' 'CARES']

Date ranges for each campaign (in specified order):
                 Campaign                             Date_Range
0                 FIREXAQ       17 July 2019 - 05 September 2019
1                 SEAC4RS     06 August 2013 - 23 September 2013
2                 CAMP2Ex       24 August 2019 - 05 October 2019
3                 ASIA-AQ        29 January 2024 - 01 April 2024
4                     DC3             18 May 2012 - 22 June 2012
5           DISCOVERAQ-DC            01 July 2011 - 29 July 2011
6            NAAMES(2016)             18 May 2016 - 01 June 2016
7   DISCOVERAQ-California     16 January 2013 - 06 February 2013
8            NAAMES(2017)  04 September 2017 - 19 September 2017
9        DISCOV